# LMNN vs NCA vs KNN — Analysis Notebook

Interactive walkthrough. Run from the project root or `notebooks/` directory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Ready.')

## 1. Dataset Overview

In [ ]:
from src.datasets import load_dataset, ALL_LOADERS
datasets = {name: load_dataset(name) for name in ALL_LOADERS}
print(f"{'Name':<26} {'Type':<12} {'Samples':>8} {'Features':>9} {'Classes':>8}")
print('-'*66)
for name, ds in datasets.items():
    print(f"{ds['name']:<26} {ds['type']:<12} {ds['n_samples']:>8} {ds['n_features']:>9} {ds['n_classes']:>8}")

## 2. Main Evaluation

In [ ]:
from src.evaluation.evaluator import Evaluator
ev = Evaluator(test_size=0.25, random_state=42)
ev.set_default_models(k=5, n_components=6, max_iter=150)
nc_map = {'iris':3,'wine':8,'digits':8,'interleaved_gaussians':6,'anisotropic_blobs':6,'noisy_moons':4}
main_results = ev.evaluate_all(list(ALL_LOADERS.keys()), n_components_map=nc_map)

In [ ]:
from src.visualization.plot_results import plot_main_results
os.makedirs('../results/plots', exist_ok=True)
plot_main_results(main_results, save_path='../results/plots/main_results.png')
from IPython.display import Image; Image('../results/plots/main_results.png')

## 3. Ablation — k sweep (Iris)

In [ ]:
from src.evaluation.ablation import AblationStudy
from src.visualization.plot_results import plot_ablation_sweep
iris = load_dataset('iris')
study = AblationStudy(iris)
k_res = study.sweep('k', [1,3,5,7,9,11], fixed={'n_components':3,'max_iter':100})
study.print_sweep(k_res, 'k')
plot_ablation_sweep(k_res,'k',dataset_name='Iris',save_path='../results/plots/ablation_k.png')
from IPython.display import Image; Image('../results/plots/ablation_k.png')

## 4. Ablation — n_components sweep (Wine)

In [ ]:
wine = load_dataset('wine')
study2 = AblationStudy(wine)
c_res = study2.sweep('n_components',[2,4,6,8,10],fixed={'k':5,'max_iter':100})
study2.print_sweep(c_res,'n_components')
plot_ablation_sweep(c_res,'n_components',dataset_name='Wine',save_path='../results/plots/ablation_comp.png')
from IPython.display import Image; Image('../results/plots/ablation_comp.png')

## 5. Embedding Visualisation

In [ ]:
from src.models import NCAModel, LMNNModel
from src.visualization.plot_embeddings import plot_embeddings
from IPython.display import display, Image
for ds_name, nc in [('iris',3),('interleaved_gaussians',6),('anisotropic_blobs',6)]:
    ds = load_dataset(ds_name)
    safe = ds['name'].lower().replace(' ','_')
    path = f'../results/plots/embeddings_{safe}.png'
    plot_embeddings(dataset=ds,
                    nca_model=NCAModel(k=5,n_components=nc,max_iter=150),
                    lmnn_model=LMNNModel(k=5,n_components=nc,max_iter=150),
                    save_path=path)
    display(Image(path))